# TF-IDF Vectorization

In [ ]:
import glob
import os
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd

## Read in the Data Files

In [ ]:
# https://archive.ics.uci.edu/dataset/311/sentence+classification 
input_folder = "../data/labeled_articles/"
file_list=glob.glob("../data/labeled_articles/*.txt")

dfs = []
for filename in file_list:
    #print(f"Processing file: {filename}")
    df = pd.read_csv(filename, delimiter='\t', names=['label', 'text'], index_col=None, header=None, on_bad_lines='skip')
    df['file'] = os.path.basename(filename)
    if df['text'].isnull().any():
        #print(f"Missing values found in file: {filename}")
        df['text']=df["label"].str[5:]  # Fix for files with missing text values
        df['label']=df["label"].str[:4]
    dfs.append(df)

df = pd.concat(dfs, ignore_index=True)
print(df.count())
print(df[df["text"].isnull()]["file"].unique().tolist())

## Build TF-IDF Vectors

In [ ]:
tfidf_vectorizer = TfidfVectorizer(stop_words='english', ngram_range=(1, 3), max_features=1000)
tfidf_vectors = tfidf_vectorizer.fit_transform(df['text']) 
idf = pd.DataFrame({'feature_name':tfidf_vectorizer.get_feature_names_out(), 
'idf_weights':tfidf_vectorizer.idf_})
idf.sort_values('idf_weights', ascending=False)

In [ ]:
## Output the vector
pd.DataFrame(tfidf_vectors.todense(), columns=tfidf_vectorizer.get_feature_names_out())